In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [ ]:

from utils.enums import Ticker, TimeFrame
from datetime import datetime
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.pyplot as plt
from utils.enums import Ticker
from feature_extraction.feature_extractor_class import FeatureExtractor
from eda.feature_explorer import FeatureExplorer


In [ ]:
# Example 1: Using bias_node_specs (recommended for feature extraction)
bias_node_specs_ts = [
    {
        'module_name': 'ts_feature',
        'timeframes': [TimeFrame.D],
        'params': {
            'wrapped_module': 'rsi',
            'wrapped_params': {'lookback': 2},  # RSI lookback period
            'lookback': 5,  # Time series feature lookback window
            'transformation': 'count_below_mean',  # From utils.functime
            'transformation_name': 'countBelowMean'
        }
    }
]

In [ ]:


# Step 1: Extract features using FeatureExtractor
extractor = FeatureExtractor(
    tickers=[
        Ticker.ES,    # E-Mini S&P
        Ticker.NQ,    # E-Mini Nasdaq
        Ticker.YM,    # E-Mini Dow
        Ticker.RTY,   # E-Mini Russell 2000
        # Ticker.HO, 
        # Ticker.GC, 
        # Ticker.SI, 
        # Ticker.CL
        # Add more tickers as needed...
    ],
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31)
)

bias_node_specs = []

windows = range(2,5,1)
avg_windows = range(2,5,1)


for lookback in windows:
    for avg_period in avg_windows:
        bias_node_specs.append({
            'module_name': 'cumulative_rsi',
            'params': {
                'lookback': lookback,
                'avg_period': avg_period
            }
        })


# roc_windows = [2,5,10,20,40,60,80, 100, 120, 140, 160, 180, 200, 252]
# for window in roc_windows:
#     bias_node_specs.append({
#         'module_name': 'roc',
#         'params': {
#             'lookback': window,
#             'normalize_by_ewsd': True,
#         }
#     })
    
#     bias_node_specs.append({
#         'module_name': 'rsi',
#         'params': {
#             'lookback': window,
#         }
#     })



# ewmac = [(16,64), (32,128), (64,256)]
# for spanFast, spanSlow in ewmac:
#     bias_node_specs.append({
#         'module_name': 'ewmac',
#         'params': {
#             'spanFast': spanFast,
#             'spanSlow': spanSlow
#         }
#     })

# Extract features
result = extractor.extract(
    bias_node_specs=bias_node_specs
)

features = result['features']              # Clean features only
normalization = result['normalization']    # ATR/EWSD columns
targets = result['targets']    

# Step 2: Create FeatureExplorer from extracted dataframes
explorer = FeatureExplorer(
    features_df=features,
    targets_df=targets, 
    metadata=result
)




In [ ]:
# Use custom model and metric
from feature_selection.base_models import QuantileBinningModel
from feature_selection.base_models import DecisionTreeBinningModel
from metrics.performance.sortino import SortinoRatio

model = QuantileBinningModel(n_bins=4, selection_metric='sortino')
metric = SortinoRatio(annualization_factor=252)
decision_tree_model = DecisionTreeBinningModel(n_bins=4, selection_metric='sortino')



In [ ]:
features.columns

In [ ]:

# Step 3: Analyze (same API as before!)
# Plot deciles with log_return
figs = explorer.plot_all_deciles(n_bins=4, target_col='log_return')
plt.show()

# figures = explorer.plot_all_uniform_bins(
#     n_bins=10,
#     target_col='log_return',
# )
# plt.show()

# # # Plot 2-bin for each feature
# for feature_name in explorer.feature_names:
#     explorer.plot_2bin(feature_name, target_col='log_return')
# plt.show()

In [ ]:

# 1. Feature-Target correlations (Spearman - captures monotonic relationships)
fig1 = explorer.plot_feature_target_correlations(
    target_col='log_return',
    figsize=(10, 8),
    annot=True  # Show correlation values
)
plt.show()

# 2. Intra-Feature correlations (Pearson - linear relationships)
fig2 = explorer.plot_feature_correlations(
    figsize=(12, 10),
    annot=True, 
    mask_diagonal=True  
)
plt.show()

In [ ]:
explorer.plot_distributions(bins=50)
plt.show()

explorer.plot_timeseries(rolling_window=50)
plt.show()

In [ ]:

results = explorer.run_permutation_test(
    target_col='log_return',
    base_model=model,
    metric=metric,
    nreps=200
)

In [ ]:
figs, summary = explorer.plot_signal_cumsum(
    target_col='log_return',
    base_model=model,
    show_plot=True,
    metric=metric,
    strategy='long-short'

)

In [ ]:
df, fig = explorer.plot_parameter_sensitivity(
    module_name='cumulative_rsi',
    param_name='lookback',
    feature_name='signal',
    target_col='log_return', 
    base_model=model,
    custom_metric=metric
)


In [ ]:
# New 2D parameter sensitivity with 3D surface plot
df_2d, fig_2d = explorer.plot_2d_parameter_surface(
    module_name='ewmac',
    param1_name='spanFast',
    param2_name='spanSlow',
    feature_name='signal',
    target_col='log_return', 
    base_model=model,
    custom_metric=metric
)